# 🎬 Talking Head Generator
**Powered by SadTalker + Starfelt monitoring**

Drop a photo + type your text → get a video of that person saying it, with an AI GENERATED watermark burned in.

> Make sure you're on a **T4 GPU**: Runtime → Change runtime type → T4 GPU

## Step 1 — Install dependencies

In [ ]:
# Install Starfelt
!pip install git+https://github.com/victorachede/starfelt.git -q

# Install gTTS for text-to-speech
!pip install gTTS -q

# Install moviepy for watermark
!pip install moviepy==1.0.3 -q

print('✅ Base deps installed')

## Step 2 — Clone & install SadTalker

In [ ]:
import os

if not os.path.exists('SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git

os.chdir('SadTalker')

!pip install -r requirements.txt -q

print('✅ SadTalker ready')

## Step 3 — Download SadTalker checkpoints

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('gfpgan/weights', exist_ok=True)

# Download main checkpoints
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -O checkpoints/mapping_00109-model.pth.tar
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -O checkpoints/mapping_00229-model.pth.tar
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -O checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors -O checkpoints/SadTalker_V0.0.2_512.safetensors

# Download GFPGAN enhancer weights
!wget -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -O gfpgan/weights/alignment_WFLW_4HG.pth
!wget -q https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -O gfpgan/weights/detection_Resnet50_Final.pth

print('✅ Checkpoints downloaded')

## Step 4 — Upload your photo & type your text

In [ ]:
from google.colab import files
from IPython.display import display, Image
import os

print('Upload your photo (clear, front-facing portrait works best):')
uploaded = files.upload()

image_path = list(uploaded.keys())[0]
print(f'\n✅ Got image: {image_path}')
display(Image(image_path, width=200))

In [ ]:
# ✏️ Type your text here
TEXT = "Hello, this is an AI-generated video. Please use this technology responsibly."

print(f'Text: "{TEXT}"')
print(f'Characters: {len(TEXT)}')

## Step 5 — Convert text to speech

In [ ]:
from gtts import gTTS
from IPython.display import Audio

AUDIO_PATH = '/content/SadTalker/input_audio.mp3'

tts = gTTS(text=TEXT, lang='en', slow=False)
tts.save(AUDIO_PATH)

print('✅ Audio generated')
Audio(AUDIO_PATH)

## Step 6 — Run SadTalker (tracked by Starfelt)

In [ ]:
import subprocess
import os
import json
import time

# Write inference script for Starfelt to wrap
inference_script = f'''
import subprocess
import sys

result = subprocess.run([
    "python", "inference.py",
    "--driven_audio", "{AUDIO_PATH}",
    "--source_image", "{image_path}",
    "--result_dir", "./results",
    "--still",
    "--preprocess", "full",
    "--enhancer", "gfpgan"
], capture_output=False)

sys.exit(result.returncode)
'''

with open('sadtalker_inference.py', 'w') as f:
    f.write(inference_script)

# Init starfelt if not already
if not os.path.exists('starfelt.yaml'):
    !starfelt init

print('🚀 Running SadTalker via Starfelt...')
print('This takes 2-5 mins. Watch the logs below.\n')

start = time.time()
result = subprocess.run(
    ['starfelt', 'run', 'sadtalker_inference.py'],
    capture_output=False
)
elapsed = time.time() - start

print(f'\n⏱️ Finished in {elapsed:.1f}s')

## Step 7 — Burn AI GENERATED watermark

In [ ]:
import glob
import os
from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip

# Find the output video
output_videos = glob.glob('./results/**/*.mp4', recursive=True)

if not output_videos:
    print('❌ No video found. Check logs above for errors.')
else:
    raw_video_path = sorted(output_videos)[-1]
    print(f'Found video: {raw_video_path}')

    # Load video
    clip = VideoFileClip(raw_video_path)

    # Create bold watermark text
    watermark = TextClip(
        '⚠ AI GENERATED',
        fontsize=28,
        color='white',
        font='DejaVu-Sans-Bold',
        stroke_color='black',
        stroke_width=2
    ).set_duration(clip.duration)

    # Position: bottom center
    watermark = watermark.set_position(('center', 'bottom')).margin(bottom=12, opacity=0)

    # Composite
    final = CompositeVideoClip([clip, watermark])

    FINAL_PATH = '/content/ai_generated_video.mp4'
    final.write_videofile(FINAL_PATH, codec='libx264', audio_codec='aac', verbose=False, logger=None)

    print(f'✅ Watermarked video saved: {FINAL_PATH}')

## Step 8 — Download your video

In [ ]:
from google.colab import files
from IPython.display import HTML

# Preview
HTML(f'''
<video width="400" controls>
  <source src="{FINAL_PATH}" type="video/mp4">
</video>
''')

In [ ]:
files.download(FINAL_PATH)
print('✅ Downloading...')

## Step 9 — Check Starfelt run stats

In [ ]:
# See cost, GPU util, duration for this run
!starfelt status
print('\n--- Full run history ---')
!starfelt cost

In [ ]:
# Inspect the last run in detail
import subprocess, json, re

status_out = subprocess.run(['starfelt', 'status'], capture_output=True, text=True).stdout
run_ids = re.findall(r'run_[a-z0-9]+', status_out)

if run_ids:
    !starfelt inspect {run_ids[-1]}
else:
    print('No run IDs found in status output. Run: starfelt status')